# Notebook 02 — Nettoyage et préparation des données Olist

**Responsable :** Yassin ADAGHOUR — Personne 1  
**Semaine 2 :** nettoyage, contrôle qualité et préparation ETL.

## Objectifs
- Diagnostiquer les données brutes.
- Réutiliser les fonctions de `src/etl/clean.py`.
- Comparer les résultats avant/après.
- Exporter les tables préparées dans `data/processed/`.
- Vérifier que les exports peuvent être relus.

> Ce notebook orchestre les fonctions du module existant. Il ne réécrit pas leur logique.

## 1. Configuration et import des fonctions

Le notebook cherche la racine du projet à partir du dossier courant. La racine doit contenir `src/etl/clean.py` et `data/raw/`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display

cwd = Path.cwd().resolve()
candidats = [cwd, *cwd.parents]
ROOT_DIR = next(
    (p for p in candidats if (p / "src" / "etl" / "clean.py").exists()
     and (p / "data" / "raw").exists()),
    cwd
)
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("Racine :", ROOT_DIR)
print("Dossier brut :", RAW_DIR, "| existe :", RAW_DIR.exists())
print("Dossier de sortie :", PROCESSED_DIR)

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Dossier data/raw introuvable : {RAW_DIR}")

from src.etl.clean import (
    diagnostiquer_qualite,
    resumer_valeurs_manquantes,
    compter_doublons_complets,
    verifier_cle_primaire,
    verifier_integrite_referentielle,
    verifier_valeurs_numeriques,
    verifier_coherence_dates_commandes,
    preparer_toutes_tables,
    exporter_tables_propres,
    valider_export,
    generer_rapport_comparaison,
    CLES_PRIMAIRES_ATTENDUES,
)
print("Import de src.etl.clean : OK")

Racine : C:\Users\PC\OneDrive\Desktop\olist-data-engineerin
Dossier brut : C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\raw | existe : True
Dossier de sortie : C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed
Import de src.etl.clean : OK


## 2. Chargement des 9 tables CSV

Les noms ci-dessous sont les clés canoniques attendues par `clean.py`, associées aux noms de fichiers Olist usuels. Le notebook vérifie les fichiers présents et ne modifie pas `data/raw/`.

In [2]:
FICHIERS = {
    "olist_orders": "olist_orders_dataset.csv",
    "olist_order_items": "olist_order_items_dataset.csv",
    "olist_customers": "olist_customers_dataset.csv",
    "olist_order_payments": "olist_order_payments_dataset.csv",
    "olist_order_reviews": "olist_order_reviews_dataset.csv",
    "olist_products": "olist_products_dataset.csv",
    "olist_sellers": "olist_sellers_dataset.csv",
    "olist_geolocation": "olist_geolocation_dataset.csv",
    "product_category_name_translation": "product_category_name_translation.csv",
}

tables_brutes = {}
fichiers_absents = []

for nom_table, nom_fichier in FICHIERS.items():
    chemin = RAW_DIR / nom_fichier
    if not chemin.exists():
        fichiers_absents.append(str(chemin))
        continue
    try:
        tables_brutes[nom_table] = pd.read_csv(chemin, encoding="utf-8")
    except UnicodeDecodeError:
        tables_brutes[nom_table] = pd.read_csv(chemin, encoding="latin-1")

if fichiers_absents:
    print("Fichiers absents :")
    for chemin in fichiers_absents:
        print("-", chemin)
if not tables_brutes:
    raise RuntimeError("Aucune table chargée. Vérifie les noms des CSV dans data/raw/.")

resume_chargement = pd.DataFrame([
    {"table": nom, "lignes": len(df), "colonnes": len(df.columns),
     "doublons_complets": int(df.duplicated().sum()),
     "valeurs_manquantes_total": int(df.isna().sum().sum())}
    for nom, df in tables_brutes.items()
]).sort_values("table").reset_index(drop=True)
display(resume_chargement)
print(f"Tables chargées : {len(tables_brutes)} / {len(FICHIERS)}")

,table,lignes,colonnes,doublons_complets,valeurs_manquantes_total
0,olist_customers,99441,5,0,0
1,olist_geolocation,1000163,5,261831,0
2,olist_order_items,112650,7,0,0
3,olist_order_payments,103886,5,0,0
4,olist_order_reviews,99224,7,0,145903
5,olist_orders,99441,8,0,4908
6,olist_products,32951,9,0,2448
7,olist_sellers,3095,4,0,0
8,product_category_name_translation,71,2,0,0


Tables chargées : 9 / 9


## 3. Diagnostic initial

On mesure la qualité des données avant tout traitement. Les résultats initiaux sont conservés pour la comparaison.

In [3]:
diagnostics_avant = [
    diagnostiquer_qualite(df, nom)
    for nom, df in tables_brutes.items()
]
diagnostic_global_avant = pd.concat(diagnostics_avant, ignore_index=True)
display(diagnostic_global_avant.head(30))

manquants_avant = pd.concat([
    resumer_valeurs_manquantes(df, nom)
    for nom, df in tables_brutes.items()
], ignore_index=True)
display(
    manquants_avant[manquants_avant["valeurs_manquantes"] > 0]
    .sort_values("pourcentage_manquant", ascending=False)
    .reset_index(drop=True)
)

,table,colonne,type,lignes_total,valeurs_manquantes,pct_manquantes,valeurs_distinctes,pct_distinctes,min,max,mean,median,std,q25,q75
0,olist_orders,order_id,object,99441,0,0.00,99441,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,olist_orders,customer_id,object,99441,0,0.00,99441,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,olist_orders,order_status,object,99441,0,0.00,8,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,olist_orders,order_purchase_timestamp,object,99441,0,0.00,98875,99.43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,olist_orders,order_approved_at,object,99441,160,0.16,90733,91.24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,olist_orders,order_delivered_carrier_date,object,99441,1783,1.79,81018,81.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,olist_orders,order_delivered_customer_date,object,99441,2965,2.98,95664,96.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,olist_orders,order_estimated_delivery_date,object,99441,0,0.00,459,0.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,olist_order_items,order_id,object,112650,0,0.00,98666,87.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,olist_order_items,order_item_id,int64,112650,0,0.00,21,0.02,1.00,21.00,1.1978,1.00,0.7051,1.00,1.0000


,table,colonne,valeurs_manquantes,pourcentage_manquant
0,olist_order_reviews,review_comment_title,87656,88.34
1,olist_order_reviews,review_comment_message,58247,58.70
2,olist_orders,order_delivered_customer_date,2965,2.98
3,olist_products,product_description_lenght,610,1.85
4,olist_products,product_category_name,610,1.85
5,olist_products,product_name_lenght,610,1.85
6,olist_products,product_photos_qty,610,1.85
7,olist_orders,order_delivered_carrier_date,1783,1.79
8,olist_orders,order_approved_at,160,0.16
9,olist_products,product_weight_g,2,0.01


### 3.1 Doublons complets

In [4]:
doublons_avant = pd.DataFrame([
    compter_doublons_complets(df, nom)
    for nom, df in tables_brutes.items()
]).sort_values("doublons_complets", ascending=False).reset_index(drop=True)
display(doublons_avant)

,table,doublons_complets,pourcentage_doublons,lignes_apres_dedup
0,olist_geolocation,261831,26.179,738332
1,olist_orders,0,0.000,99441
2,olist_order_items,0,0.000,112650
3,olist_order_payments,0,0.000,103886
4,olist_customers,0,0.000,99441
5,olist_order_reviews,0,0.000,99224
6,olist_products,0,0.000,32951
7,olist_sellers,0,0.000,3095
8,product_category_name_translation,0,0.000,71


### 3.2 Clés primaires candidates

In [5]:
resultats_cles_avant = []
for nom_table, colonnes in CLES_PRIMAIRES_ATTENDUES.items():
    if nom_table in tables_brutes:
        resultats_cles_avant.append(
            verifier_cle_primaire(tables_brutes[nom_table], colonnes, nom_table)
        )

if resultats_cles_avant:
    display(pd.concat(resultats_cles_avant, ignore_index=True))
else:
    print("Aucune table correspondant aux clés configurées n'a été chargée.")

,table,colonne,statut,valeurs_manquantes,valeurs_distinctes,doublons,est_unique
0,olist_customers,customer_id,OK,0,99441,0,True
1,olist_orders,order_id,OK,0,99441,0,True
2,olist_products,product_id,OK,0,32951,0,True
3,olist_sellers,seller_id,OK,0,3095,0,True
4,product_category_name_translation,product_category_name,OK,0,71,0,True


### 3.3 Intégrité référentielle

In [6]:
integrite_avant = verifier_integrite_referentielle(tables_brutes)
display(integrite_avant)

,table_source,colonne_source,table_cible,colonne_cible,statut,references_orphelines
0,olist_orders,customer_id,olist_customers,customer_id,OK,0
1,olist_order_items,order_id,olist_orders,order_id,OK,0
2,olist_order_items,product_id,olist_products,product_id,OK,0
3,olist_order_items,seller_id,olist_sellers,seller_id,OK,0
4,olist_order_payments,order_id,olist_orders,order_id,OK,0
5,olist_order_reviews,order_id,olist_orders,order_id,OK,0


## 4. Préparation des tables avec `preparer_toutes_tables()`

Cette fonction réutilise les traitements centralisés dans `src/etl/clean.py`. Les valeurs manquantes métier peuvent être conservées volontairement. Les valeurs extrêmes sont signalées, et non supprimées automatiquement.

In [7]:
tables_propres, metrics_preparation = preparer_toutes_tables(tables_brutes)
print("Tables préparées :", len(tables_propres))
display(pd.DataFrame(metrics_preparation))

2026-10-27 10:01:56,096 - INFO - === Préparation de olist_customers ===
2026-10-27 10:01:56,487 - INFO - === olist_customers terminé : 99441 -> 99441 lignes ===
2026-10-27 10:01:56,489 - INFO - === Préparation de olist_products ===
2026-10-27 10:01:56,514 - INFO - [olist_products] product_category_name: 610 valeurs manquantes imputées par 'inconnu'
2026-10-27 10:01:56,547 - INFO - [olist_products] product_name_lenght: 610 valeurs manquantes imputées (médiane)
2026-10-27 10:01:56,564 - INFO - [olist_products] product_description_lenght: 610 valeurs manquantes imputées (médiane)
2026-10-27 10:01:56,613 - INFO - [olist_products] product_photos_qty: 610 valeurs manquantes imputées (médiane)
2026-10-27 10:01:56,628 - INFO - [olist_products] product_weight_g: 2 valeurs manquantes imputées (médiane)
2026-10-27 10:01:56,646 - INFO - [olist_products] product_length_cm: 2 valeurs manquantes imputées (médiane)
2026-10-27 10:01:56,663 - INFO - [olist_products] product_height_cm: 2 valeurs manquant

Tables préparées : 9


,table,lignes_initial,colonnes_initial,verification_cles,lignes_final,colonnes_final,lignes_supprimees,anomalies_numeriques,dates_conversion_echecs
0,olist_customers,99441,5,"[{'table': 'olist_customers', 'colonne': 'cust...",99441,5,0,NaN,NaN
1,olist_products,32951,9,"[{'table': 'olist_products', 'colonne': 'produ...",32951,9,0,"{'product_weight_g': {'extremes_IQR': 4551}, '...",NaN
2,olist_sellers,3095,4,"[{'table': 'olist_sellers', 'colonne': 'seller...",3095,4,0,NaN,NaN
3,product_category_name_translation,71,2,[{'table': 'product_category_name_translation'...,71,2,0,NaN,NaN
4,olist_orders,99441,8,"[{'table': 'olist_orders', 'colonne': 'order_i...",99441,8,0,NaN,"{'order_purchase_timestamp': 0, 'order_approve..."
5,olist_order_items,112650,7,NaN,112650,7,0,"{'price': {'extremes_IQR': 8427}, 'freight_val...",{'shipping_limit_date': 0}
6,olist_order_payments,103886,5,NaN,103886,5,0,"{'payment_value': {'extremes_IQR': 7981}, 'pay...",NaN
7,olist_order_reviews,99224,7,NaN,99224,7,0,{},"{'review_creation_date': 0, 'review_answer_tim..."
8,olist_geolocation,1000163,5,NaN,738332,5,261831,NaN,NaN


### 4.1 Contrôle de cohérence des dates de commandes

In [8]:
if "olist_orders" in tables_propres:
    incoherences_dates = verifier_coherence_dates_commandes(tables_propres["olist_orders"])
    display(pd.DataFrame([
        {"controle": nom, "nombre_de_cas": nb}
        for nom, nb in incoherences_dates.items()
    ]))
else:
    print("Table olist_orders indisponible.")

2026-10-27 10:02:15,169 - WARNING - [olist_orders] Incohérence 'livre_transporteur_avant_approuve': 1359 cas
2026-10-27 10:02:15,170 - WARNING - [olist_orders] Incohérence 'livre_client_avant_transporteur': 23 cas
2026-10-27 10:02:15,173 - WARNING - [olist_orders] Incohérence 'retard_livraison': 7827 cas


,controle,nombre_de_cas
0,approuve_avant_achat,0
1,livre_transporteur_avant_approuve,1359
2,livre_client_avant_transporteur,23
3,estimee_avant_achat,0
4,retard_livraison,7827


### 4.2 Résumé des anomalies numériques

In [9]:
anomalies_numeriques = []
for metric in metrics_preparation:
    for colonne, details in metric.get("anomalies_numeriques", {}).items():
        for type_anomalie, nombre in details.items():
            anomalies_numeriques.append({
                "table": metric["table"], "colonne": colonne,
                "anomalie": type_anomalie, "nombre": nombre
            })

if anomalies_numeriques:
    display(pd.DataFrame(anomalies_numeriques))
else:
    print("Aucune anomalie numérique détectée par les règles configurées.")

,table,colonne,anomalie,nombre
0,olist_products,product_weight_g,extremes_IQR,4551
1,olist_products,product_length_cm,extremes_IQR,1380
2,olist_products,product_height_cm,extremes_IQR,1892
3,olist_products,product_width_cm,extremes_IQR,912
4,olist_products,product_description_lenght,extremes_IQR,2174
5,olist_products,product_photos_qty,extremes_IQR,849
6,olist_order_items,price,extremes_IQR,8427
7,olist_order_items,freight_value,extremes_IQR,11613
8,olist_order_payments,payment_value,extremes_IQR,7981
9,olist_order_payments,payment_installments,extremes_IQR,6313


## 5. Comparaison avant/après

In [10]:
comparaison = generer_rapport_comparaison(
    tables_brutes=tables_brutes,
    tables_propres=tables_propres,
    metrics=metrics_preparation
)
display(comparaison)

,table,lignes_avant,lignes_apres,lignes_supprimees,colonnes_avant,colonnes_apres,doublons_complets_avant,doublons_complets_apres,valeurs_manquantes_avant,valeurs_manquantes_apres,dates_conversion_echecs,anomalies_numeriques,incoherences_dates
0,olist_customers,99441,99441,0,5,5,0,0,0,0,{},{},{}
1,olist_products,32951,32951,0,9,9,0,0,2448,0,{},"{'product_weight_g': {'extremes_IQR': 4551}, '...",{}
2,olist_sellers,3095,3095,0,4,4,0,0,0,0,{},{},{}
3,product_category_name_translation,71,71,0,2,2,0,0,0,0,{},{},{}
4,olist_orders,99441,99441,0,8,8,0,0,4908,4908,"{'order_purchase_timestamp': 0, 'order_approve...",{},{}
5,olist_order_items,112650,112650,0,7,7,0,0,0,0,{'shipping_limit_date': 0},"{'price': {'extremes_IQR': 8427}, 'freight_val...",{}
6,olist_order_payments,103886,103886,0,5,5,0,0,0,0,{},"{'payment_value': {'extremes_IQR': 7981}, 'pay...",{}
7,olist_order_reviews,99224,99224,0,7,7,0,0,145903,145903,"{'review_creation_date': 0, 'review_answer_tim...",{},{}
8,olist_geolocation,1000163,738332,261831,5,5,261831,0,0,0,{},{},{}


## 6. Export dans `data/processed/`

Les tables sont exportées avec le suffixe `_clean`. Les fichiers originaux dans `data/raw/` ne sont jamais écrasés.

In [11]:
fichiers_exportes = exporter_tables_propres(
    tables=tables_propres,
    dossier_sortie=PROCESSED_DIR,
    suffixe="_clean",
    encoding="utf-8"
)
print("Nombre de fichiers exportés :", len(fichiers_exportes))
for chemin in fichiers_exportes:
    print("-", chemin)

2026-10-27 10:02:40,283 - INFO - Exporté : C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_customers_clean.csv (99,441 lignes × 5 colonnes)
2026-10-27 10:02:41,398 - INFO - Exporté : C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_products_clean.csv (32,951 lignes × 9 colonnes)
2026-10-27 10:02:41,472 - INFO - Exporté : C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_sellers_clean.csv (3,095 lignes × 4 colonnes)
2026-10-27 10:02:41,482 - INFO - Exporté : C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\product_category_name_translation_clean.csv (71 lignes × 2 colonnes)
2026-10-27 10:02:45,784 - INFO - Exporté : C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_orders_clean.csv (99,441 lignes × 8 colonnes)
2026-10-27 10:02:48,218 - INFO - Exporté : C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_order_items_clean.csv (112,650 lignes × 7 colonnes)
2026-1

Nombre de fichiers exportés : 9
- C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_customers_clean.csv
- C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_products_clean.csv
- C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_sellers_clean.csv
- C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\product_category_name_translation_clean.csv
- C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_orders_clean.csv
- C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_order_items_clean.csv
- C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_order_payments_clean.csv
- C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_order_reviews_clean.csv
- C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed\olist_geolocation_clean.csv


## 7. Validation des exports

`valider_export()` compare notamment les dimensions, les colonnes et les types. Après un export CSV, les colonnes de dates peuvent être relues comme chaînes de caractères : une différence de type doit donc être interprétée avec prudence.

In [12]:
validation_exports = valider_export(
    tables=tables_propres,
    dossier_sortie=PROCESSED_DIR,
    suffixe="_clean"
)
display(validation_exports)

,table,statut,lignes_original,lignes_rechargees,colonnes_original,colonnes_rechargees,identique
0,olist_customers,OK,99441,99441,5,5,True
1,olist_products,OK,32951,32951,9,9,True
2,olist_sellers,OK,3095,3095,4,4,True
3,product_category_name_translation,OK,71,71,2,2,True
4,olist_orders,DIFFERENT,99441,99441,8,8,False
5,olist_order_items,DIFFERENT,112650,112650,7,7,False
6,olist_order_payments,OK,103886,103886,5,5,True
7,olist_order_reviews,DIFFERENT,99224,99224,7,7,False
8,olist_geolocation,OK,738332,738332,5,5,True


### 7.1 Vérification indépendante de la relecture

In [13]:
controles_relecture = []
for nom_table, df_original in tables_propres.items():
    chemin = PROCESSED_DIR / f"{nom_table}_clean.csv"
    ligne = {
        "table": nom_table, "fichier_existe": chemin.exists(),
        "lecture_ok": False, "lignes_attendues": len(df_original),
        "lignes_relues": None, "colonnes_identiques": False, "statut": "ERREUR"
    }
    if chemin.exists():
        try:
            df_relue = pd.read_csv(chemin, encoding="utf-8")
            ligne["lecture_ok"] = True
            ligne["lignes_relues"] = len(df_relue)
            ligne["colonnes_identiques"] = list(df_relue.columns) == list(df_original.columns)
            ligne["statut"] = (
                "OK" if len(df_relue) == len(df_original)
                and ligne["colonnes_identiques"]
                else "DIMENSIONS_OU_COLONNES_DIFFERENTES"
            )
        except Exception as exc:
            ligne["statut"] = f"ERREUR_LECTURE: {exc}"
    controles_relecture.append(ligne)

display(pd.DataFrame(controles_relecture))

,table,fichier_existe,lecture_ok,lignes_attendues,lignes_relues,colonnes_identiques,statut
0,olist_customers,True,True,99441,99441,True,OK
1,olist_products,True,True,32951,32951,True,OK
2,olist_sellers,True,True,3095,3095,True,OK
3,product_category_name_translation,True,True,71,71,True,OK
4,olist_orders,True,True,99441,99441,True,OK
5,olist_order_items,True,True,112650,112650,True,OK
6,olist_order_payments,True,True,103886,103886,True,OK
7,olist_order_reviews,True,True,99224,99224,True,OK
8,olist_geolocation,True,True,738332,738332,True,OK


## 8. Diagnostic après préparation

In [14]:
diagnostics_apres = [
    diagnostiquer_qualite(df, nom)
    for nom, df in tables_propres.items()
]
diagnostic_global_apres = pd.concat(diagnostics_apres, ignore_index=True)
display(
    diagnostic_global_apres[diagnostic_global_apres["valeurs_manquantes"] > 0]
    .sort_values(["table", "pct_manquantes"], ascending=[True, False])
)

,table,colonne,type,lignes_total,valeurs_manquantes,pct_manquantes,valeurs_distinctes,pct_distinctes,min,max,mean,median,std,q25,q75
43,olist_order_reviews,review_comment_title,object,99224,87656,88.34,4527,4.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44,olist_order_reviews,review_comment_message,object,99224,58247,58.70,36159,36.44,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,olist_orders,order_delivered_customer_date,datetime64[ns],99441,2965,2.98,95664,96.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,olist_orders,order_delivered_carrier_date,datetime64[ns],99441,1783,1.79,81018,81.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,olist_orders,order_approved_at,datetime64[ns],99441,160,0.16,90733,91.24,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 9. Synthèse avant/après

In [15]:
synthese_finale = pd.DataFrame([
    {
        "table": nom,
        "lignes_brutes": len(tables_brutes[nom]),
        "lignes_preparees": len(tables_propres[nom]),
        "colonnes_brutes": len(tables_brutes[nom].columns),
        "colonnes_preparees": len(tables_propres[nom].columns),
        "doublons_bruts": int(tables_brutes[nom].duplicated().sum()),
        "doublons_prepares": int(tables_propres[nom].duplicated().sum()),
        "valeurs_manquantes_brutes": int(tables_brutes[nom].isna().sum().sum()),
        "valeurs_manquantes_preparees": int(tables_propres[nom].isna().sum().sum())
    }
    for nom in tables_propres if nom in tables_brutes
])
display(synthese_finale)

print("Tables chargées :", len(tables_brutes))
print("Tables préparées :", len(tables_propres))
print("Fichiers exportés :", len(fichiers_exportes) if "fichiers_exportes" in globals() else 0)
print("Dossier de sortie :", PROCESSED_DIR)

,table,lignes_brutes,lignes_preparees,colonnes_brutes,colonnes_preparees,doublons_bruts,doublons_prepares,valeurs_manquantes_brutes,valeurs_manquantes_preparees
0,olist_customers,99441,99441,5,5,0,0,0,0
1,olist_products,32951,32951,9,9,0,0,2448,0
2,olist_sellers,3095,3095,4,4,0,0,0,0
3,product_category_name_translation,71,71,2,2,0,0,0,0
4,olist_orders,99441,99441,8,8,0,0,4908,4908
5,olist_order_items,112650,112650,7,7,0,0,0,0
6,olist_order_payments,103886,103886,5,5,0,0,0,0
7,olist_order_reviews,99224,99224,7,7,0,0,145903,145903
8,olist_geolocation,1000163,738332,5,5,261831,0,0,0


Tables chargées : 9
Tables préparées : 9
Fichiers exportés : 9
Dossier de sortie : C:\Users\PC\OneDrive\Desktop\olist-data-engineerin\data\processed


## 10. Conclusion

Compléter l'interprétation à partir des résultats réellement affichés après exécution :

- Les traitements appliqués sont ceux documentés dans `src/etl/clean.py`.
- Les doublons complets sont supprimés uniquement dans la table de géolocalisation par la fonction dédiée.
- Certaines valeurs manquantes sont conservées lorsqu'elles ont une signification métier, notamment les dates des commandes non livrées et les commentaires facultatifs des avis.
- Les valeurs extrêmes sont signalées par les contrôles numériques, mais ne sont pas automatiquement supprimées.
- Les anomalies restantes doivent être examinées avant de calculer les KPI.

**Important :** ce notebook n'est pas pré-exécuté ici, car il dépend de ton dossier de projet et de tes 9 CSV locaux. Exécute les cellules dans l'ordre depuis ton projet pour obtenir les résultats réels.